# 02 — Training & Evaluation

**Project:** Post-Only (A) vs Trajectory (B) Supervision for Continual Tool-Use Learning

This notebook trains on 6 sequential domain blocks and evaluates after each.

In [1]:
# ============================================================
# CONFIGURATION
# ============================================================
CONDITION = "B"   # "A", "B", or "A+"
SEED = 42

In [2]:
!pip install -q transformers accelerate peft bitsandbytes trl huggingface_hub tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 91.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 73.8 MB/s eta 0:00:00


In [3]:
import json
import os
import re
import random
import time
import pickle
import numpy as np
from tqdm.auto import tqdm

import torch
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    TrainingArguments, DataCollatorForLanguageModeling, Trainer,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from torch.utils.data import Dataset

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print(f"Condition: {CONDITION} | Seed: {SEED}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

Condition: B | Seed: 42
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


AttributeError: 'torch._C._CudaDeviceProperties' object has no attribute 'total_mem'

## 1. Load Preprocessed Data

In [ ]:
with open('preprocessed_data/preprocessed.pkl', 'rb') as f:
    data = pickle.load(f)

blocks = data['blocks']
config = data['config']
MODEL_NAME = config['model_name']
MAX_SEQ_LEN = config['max_seq_len']
NUM_BLOCKS = config['num_blocks']
BASE_EPOCHS = config['base_epochs']

print(f"Model: {MODEL_NAME}")
print(f"Blocks: {NUM_BLOCKS}, Seq len: {MAX_SEQ_LEN}")
print(f"\nBlock sizes:")
for b in blocks:
    print(f"  D{b['block_id']}: {len(b['train_a'])} train, {len(b['eval_a'])} eval")

In [ ]:
def get_train_texts(block, condition):
    if condition in ('A', 'A+'):
        return block['train_a'], block['train_a_prompt_lens']
    return block['train_b'], block['train_b_prompt_lens']

def get_eval_texts(block, condition):
    if condition in ('A', 'A+'):
        return block['eval_a'], block['eval_a_prompt_lens']
    return block['eval_b'], block['eval_b_prompt_lens']

def get_epochs(block, condition):
    if condition == 'A+':
        return block['aplus_epochs']
    return BASE_EPOCHS

## 2. Load Model

In [ ]:
ATTN_IMPL = "sdpa"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def load_fresh_model():
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        attn_implementation=ATTN_IMPL,
    )
    model = prepare_model_for_kbit_training(model)
    lora_config = LoraConfig(
        r=32, lora_alpha=64,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model

model = load_fresh_model()

## 3. Dataset & Training

In [ ]:
class TextDataset(Dataset):
    # Dataset with prompt-masked labels for causal LM training
    def __init__(self, texts, prompt_lens, tokenizer, max_length):
        self.items = []
        for text, plen in zip(texts, prompt_lens):
            enc = tokenizer(
                text, truncation=True, max_length=max_length,
                padding="max_length", return_tensors="pt",
            )
            input_ids = enc['input_ids'].squeeze()
            attn_mask = enc['attention_mask'].squeeze()
            labels = input_ids.clone()
            labels[:min(plen, max_length)] = -100  # mask prompt
            labels[attn_mask == 0] = -100  # mask padding
            self.items.append({
                'input_ids': input_ids,
                'attention_mask': attn_mask,
                'labels': labels,
            })

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        return self.items[idx]

In [ ]:
BATCH_SIZE = 16
LR = 2e-4

def train_on_block(model, texts, prompt_lens, block_name, num_epochs):
    # Train model on one block. Returns (loss, elapsed).
    dataset = TextDataset(texts, prompt_lens, tokenizer, MAX_SEQ_LEN)
    print(f"  Training {len(dataset)} examples, {num_epochs} epochs...")

    args = TrainingArguments(
        output_dir=f"/tmp/ckpt_{block_name}",
        num_train_epochs=num_epochs,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=1,
        learning_rate=LR,
        bf16=True,
        logging_steps=10,
        save_strategy="no",
        report_to="none",
        optim="paged_adamw_8bit",
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        seed=SEED,
        dataloader_pin_memory=True,
        dataloader_num_workers=4,
        gradient_checkpointing=False,
    )

    trainer = Trainer(
        model=model, train_dataset=dataset, args=args,
        data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    )
    start = time.time()
    result = trainer.train()
    elapsed = time.time() - start
    print(f"  {block_name}: loss={result.training_loss:.4f}, time={elapsed:.0f}s")
    return result.training_loss, elapsed

## 4. Evaluation Functions

In [ ]:
@torch.no_grad()
def evaluate_loss(model, texts, prompt_lens, max_samples=100):
    # Compute average loss and perplexity on response tokens only
    model.eval()
    indices = random.sample(range(len(texts)), min(max_samples, len(texts)))
    total_loss, total_tokens = 0.0, 0

    for idx in indices:
        enc = tokenizer(
            texts[idx], truncation=True,
            max_length=MAX_SEQ_LEN, return_tensors="pt",
        ).to(model.device)
        labels = enc['input_ids'].clone()
        plen = min(prompt_lens[idx], labels.shape[1])
        labels[0, :plen] = -100
        labels[enc['attention_mask'] == 0] = -100

        outputs = model(
            input_ids=enc['input_ids'],
            attention_mask=enc['attention_mask'],
            labels=labels,
        )
        n = (labels != -100).sum().item()
        if n > 0:
            total_loss += outputs.loss.item() * n
            total_tokens += n

    avg_loss = total_loss / total_tokens if total_tokens > 0 else float('inf')
    ppl = np.exp(min(avg_loss, 100))
    model.train()
    return avg_loss, ppl

In [ ]:
@torch.no_grad()
def evaluate_generation(model, entries, max_samples=100):
    # Evaluate API-call generation accuracy
    # Returns (api_name_accuracy, full_accuracy)
    model.eval()
    if len(entries) > max_samples:
        entries = random.sample(entries, max_samples)

    system_prompt = config['system_prompt']
    name_correct, full_correct, total = 0, 0, 0

    for entry in entries:
        expected = entry.get('output', '')
        match = re.search(r'\[([A-Za-z_][A-Za-z0-9_]*)\(', expected)
        if not match:
            continue
        expected_api = match.group(1)
        expected_params = dict(re.findall(r"(\w+)='([^']*)'", expected))

        inp = entry['input']
        prompt = f"[INST] {system_prompt}\n\n{inp} [/INST]"

        enc = tokenizer(
            prompt, truncation=True,
            max_length=MAX_SEQ_LEN - 128, return_tensors="pt",
        ).to(model.device)

        gen = model.generate(
            **enc, max_new_tokens=128,
            do_sample=False, pad_token_id=tokenizer.eos_token_id,
        )
        generated = tokenizer.decode(
            gen[0][enc['input_ids'].shape[1]:], skip_special_tokens=True
        )

        if expected_api.lower() in generated.lower():
            name_correct += 1
            if expected_params:
                gen_params = dict(re.findall(r"(\w+)='([^']*)'", generated))
                if any(
                    gen_params.get(k, '').lower() == v.lower()
                    for k, v in expected_params.items()
                ):
                    full_correct += 1
            else:
                full_correct += 1
        total += 1

    model.train()
    name_acc = name_correct / total if total > 0 else 0.0
    full_acc = full_correct / total if total > 0 else 0.0
    return name_acc, full_acc

## 5. Zero-Shot Baseline

In [ ]:
print("=" * 60)
print("ZERO-SHOT BASELINE")
print("=" * 60)

zero_shot = {'loss': [], 'ppl': [], 'name_acc': [], 'full_acc': []}

for j in range(NUM_BLOCKS):
    eval_texts, eval_plens = get_eval_texts(blocks[j], CONDITION)
    loss, ppl = evaluate_loss(model, eval_texts, eval_plens)
    name_acc, full_acc = evaluate_generation(model, blocks[j]['eval_entries_raw'])
    zero_shot['loss'].append(loss)
    zero_shot['ppl'].append(ppl)
    zero_shot['name_acc'].append(name_acc)
    zero_shot['full_acc'].append(full_acc)
    print(f"  D{j+1}: loss={loss:.3f}, ppl={ppl:.1f}, "
          f"name={name_acc:.1%}, full={full_acc:.1%}")

## 6. Continual Learning Loop

In [ ]:
print(f"\n{'=' * 60}")
print(f"CONTINUAL LEARNING — Condition {CONDITION}, Seed {SEED}")
print(f"{'=' * 60}")

eval_loss_mat = np.zeros((NUM_BLOCKS, NUM_BLOCKS))
eval_ppl_mat = np.zeros((NUM_BLOCKS, NUM_BLOCKS))
eval_acc_mat = np.zeros((NUM_BLOCKS, NUM_BLOCKS))
eval_full_acc_mat = np.zeros((NUM_BLOCKS, NUM_BLOCKS))
train_losses, train_times, epochs_per_block = [], [], []

CHECKPOINT_DIR = f"checkpoints_{CONDITION}_seed{SEED}"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
experiment_start = time.time()

for i in range(NUM_BLOCKS):
    print(f"\n--- Training on D{i+1}/{NUM_BLOCKS} ---")
    train_texts, train_plens = get_train_texts(blocks[i], CONDITION)
    epochs = get_epochs(blocks[i], CONDITION)
    epochs_per_block.append(epochs)

    t_loss, t_time = train_on_block(
        model, train_texts, train_plens,
        block_name=f"{CONDITION}_s{SEED}_D{i+1}",
        num_epochs=epochs,
    )
    train_losses.append(t_loss)
    train_times.append(t_time)

    print(f"  Evaluating all blocks...")
    for j in range(NUM_BLOCKS):
        eval_texts, eval_plens = get_eval_texts(blocks[j], CONDITION)
        loss, ppl = evaluate_loss(model, eval_texts, eval_plens)
        name_acc, full_acc = evaluate_generation(model, blocks[j]['eval_entries_raw'])
        eval_loss_mat[i][j] = loss
        eval_ppl_mat[i][j] = ppl
        eval_acc_mat[i][j] = name_acc
        eval_full_acc_mat[i][j] = full_acc
        tag = "(curr)" if j == i else "(prev)" if j < i else "(fut)"
        print(f"    D{j+1} {tag}: loss={loss:.3f}, ppl={ppl:.1f}, "
              f"name={name_acc:.1%}, full={full_acc:.1%}")

    # Checkpoint after each block
    ckpt = {
        'condition': CONDITION, 'seed': SEED,
        'blocks_trained': i + 1,
        'eval_loss': eval_loss_mat[:i+1].tolist(),
        'eval_ppl': eval_ppl_mat[:i+1].tolist(),
        'eval_acc': eval_acc_mat[:i+1].tolist(),
        'eval_full_acc': eval_full_acc_mat[:i+1].tolist(),
        'train_losses': train_losses, 'train_times': train_times,
    }
    with open(f'{CHECKPOINT_DIR}/after_D{i+1}.json', 'w') as f:
        json.dump(ckpt, f, indent=2)
    print(f"  Checkpoint saved")

total_time = time.time() - experiment_start
print(f"\nTotal: {total_time:.0f}s ({total_time/60:.1f} min)")

## 7. Save Final Results

In [ ]:
results = {
    'condition': CONDITION,
    'seed': SEED,
    'zero_shot': zero_shot,
    'eval_loss': eval_loss_mat.tolist(),
    'eval_ppl': eval_ppl_mat.tolist(),
    'eval_acc': eval_acc_mat.tolist(),
    'eval_full_acc': eval_full_acc_mat.tolist(),
    'train_losses': train_losses,
    'train_times': train_times,
    'epochs_per_block': epochs_per_block,
    'total_time': total_time,
    'config': {
        'model': MODEL_NAME,
        'num_blocks': NUM_BLOCKS,
        'base_epochs': BASE_EPOCHS,
        'batch_size': BATCH_SIZE,
        'lr': LR,
        'max_seq_len': MAX_SEQ_LEN,
        'lora_r': 32, 'lora_alpha': 64,
        'attn': ATTN_IMPL,
        'precision': 'bf16',
    },
}

output_file = f"results_{CONDITION}_seed{SEED}.json"
with open(output_file, 'w') as f:
    json.dump(results, f, indent=2)

print(f"\nResults saved: {output_file}")